In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_usage_rates(start_year, end_year, position='PF'):
    all_data = []

    for year in range(start_year, end_year + 1):
        url = f"https://www.basketball-reference.com/leagues/NBA_{year}_advanced.html"
        print(f"Scraping data for year: {year}")
        response = requests.get(url)

        # Check if the request was successful
        if response.status_code != 200:
            print(f"Failed to retrieve data for year {year}")
            continue

        soup = BeautifulSoup(response.content, 'html.parser')

        table = soup.find('table', {'id': 'advanced_stats'})

        # Debugging output for missing tables
        if table is None:
            print(f"Table not found for year {year}")
            continue

        rows = table.find_all('tr')

        for row in rows[1:]:
            cells = row.find_all('td')
            if cells:
                player_position = cells[3].text  # Position column
                if position in player_position:
                    data = [cell.text for cell in cells]
                    data.append(year)
                    all_data.append(data)

        # Be respectful to the website by adding a delay
        time.sleep(2)

    # Create DataFrame
    if all_data:
        columns = [th.text for th in table.find('thead').find_all('th')][1:]
        columns.append('Year')
        df = pd.DataFrame(all_data, columns=columns)
    else:
        df = pd.DataFrame()

    return df

# Scrape PF usage rates for 1990-2005 and 2020-2023
pf_usage_90s_00s = scrape_usage_rates(1990, 2005)
pf_usage_20s = scrape_usage_rates(2020, 2023)

# Save to CSV if DataFrames are not empty
if not pf_usage_90s_00s.empty:
    pf_usage_90s_00s.to_csv('pf_usage_90s_00s.csv', index=False)
    print("Saved pf_usage_90s_00s.csv")

if not pf_usage_20s.empty:
    pf_usage_20s.to_csv('pf_usage_20s.csv', index=False)
    print("Saved pf_usage_20s.csv")

# Print the first few rows of each DataFrame
print(pf_usage_90s_00s.head())
print(pf_usage_20s.head())
